# Prerequisites

In [89]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

File ‘around_the_world_in_80_days.txt’ already there; not retrieving.


# 1. Word Count

Instructions:
For each cell marked "double-click and add explanation here" please answer the question in your own words.
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code.

In [90]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [91]:
# Define the rdd (relative path: the file was downloaded next to the notebook)
rdd = sc.textFile('around_the_world_in_80_days.txt')

In [92]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [93]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [94]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:59

double-click and add explanation here

The output is not a list of words, it's just an object PythonRDD. That's because flatMap is a transformation and spark is lazy : it doesn't open the document and doesn't split anything yet, it only write in the DAG that words = rdd splitted by space. The real work will be done later when we call an action.

In [95]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for',
 'the',
 'use',
 'of',
 'anyone',
 'anywhere',
 'in',
 'the',
 'United',
 'States',
 'and',
 'most',
 'other',
 'parts',
 'of',
 'the',
 'world',
 'at',
 'no',
 'cost',
 'and',
 'with',
 'almost',
 'no',
 'restrictions',
 'whatsoever.',
 'You',
 'may',
 'copy',
 'it,',
 'give',
 'it',
 'away',
 'or',
 're-use',
 'it',
 'under',
 'the',
 'terms',
 'of',
 'the',
 'Project',
 'Gutenberg',
 'License',
 'included',
 'with',
 'this',
 'eBook',
 'or',
 'online',
 'at',
 'www.gutenberg.org.',
 'If',
 'you',
 'are',
 'not',
 'located',
 'in',
 'the',
 'United',
 'States,',
 'you',
 'will',
 'have',
 'to',
 'check',
 'the',
 'laws',
 'of',
 'the',
 'country',
 'where',
 'you',
 'are',
 'located',
 'before',
 'using',
 'this',
 'eBook.',
 '',
 'Title:',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 'Author:',
 'Jules'

double-click and add explanation here

We saw the entire list of every words which is very long, collect trigger the calculation and goes to the driver, it's the opposite of lazyness, but as said before it's very long so big volume !

In [96]:
# nicer print (limited to 30 words to keep the notebook readable)
for w in words.take(30):
    print(w)

The
Project
Gutenberg
eBook
of
Around
the
World
in
Eighty
Days





This
eBook
is
for
the
use
of
anyone
anywhere
in
the
United
States
and


In [97]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [98]:
%%time
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
print("map     :", rdd.map(lambda l: l.split(' ')).take(3))
print("flatMap :", rdd.flatMap(lambda l: l.split(' ')).take(3))

map     : [['The', 'Project', 'Gutenberg', 'eBook', 'of', 'Around', 'the', 'World', 'in', 'Eighty', 'Days'], ['', '', '', '', ''], ['This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'United', 'States', 'and']]
flatMap : ['The', 'Project', 'Gutenberg']
CPU times: user 22.7 ms, sys: 7.26 ms, total: 30 ms
Wall time: 477 ms


map give exactly the same amount in output than in input whereas produce 0 to n outputs and flat them in the rdd result

%%time calculate the time that the cell needs to execute itself

In [99]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

words.take(20)

[('The', 1),
 ('Project', 1),
 ('Gutenberg', 1),
 ('eBook', 1),
 ('of', 1),
 ('Around', 1),
 ('the', 1),
 ('World', 1),
 ('in', 1),
 ('Eighty', 1),
 ('Days', 1),
 ('', 1),
 ('', 1),
 ('', 1),
 ('', 1),
 ('', 1),
 ('This', 1),
 ('eBook', 1),
 ('is', 1),
 ('for', 1)]

In [100]:
# a. count the occurence of each word
counts = words.reduceByKey(lambda a, b: a + b)
counts.take(10)

[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10)]

In [101]:
# b. a common first step in text analysis, change all capital letters to lower case
counts_lower = rdd.flatMap(lambda line: line.lower().split(' ')) \
                  .map(lambda w: (w, 1)) \
                  .reduceByKey(lambda a, b: a + b)
counts_lower.take(10)

[('of', 1926),
 ('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('', 2193),
 ('this', 341),
 ('for', 414),
 ('use', 19),
 ('anyone', 6)]

In [102]:
# c. eliminate the stop words.
STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'at', 'by', 'for',
    'with', 'from', 'as', 'is', 'was', 'were', 'be', 'been', 'are', 'am', 'it', 'its',
    'he', 'she', 'they', 'them', 'his', 'her', 'their', 'him', 'i', 'you', 'we', 'me',
    'my', 'your', 'our', 'this', 'that', 'these', 'those', 'there', 'here', 'not', 'no',
    'so', 'if', 'then', 'than', 'which', 'who', 'whom', 'what', 'when', 'where', 'had',
    'has', 'have', 'do', 'did', 'does', 'would', 'could', 'should', 'will', 'shall',
    'may', 'might', 'can', 'up', 'out', 'into', 'over', 'all', 'any', 'some', 'very',
    'more', 'most', 'one', 'two', 'said', 'mr', 'mrs', 'himself', 'only', 'now', 'must',
    'about', 'after', 'before', 'through', 'upon', 'off', 'again', 'own', 'such',
    'other', 'each', 'while', 'being', 'made', 'like', 'much', 'well', 'go', 'went',
    # Project Gutenberg header and license, present in the file
    'project', 'gutenberg', 'ebook', 'www', 'org',
    # leftovers of contractions once the apostrophe is split: Fogg's, don't, I'll...
    's', 't', 'll', 're', 've', 'd', 'm', '',
}

counts_nostop = rdd.flatMap(lambda line: line.lower().split(' ')) \
                   .filter(lambda w: w not in STOPWORDS) \
                   .map(lambda w: (w, 1)) \
                   .reduceByKey(lambda a, b: a + b)
counts_nostop.take(10)

[('around', 32),
 ('world', 35),
 ('eighty', 27),
 ('days', 46),
 ('use', 19),
 ('anyone', 6),
 ('united', 27),
 ('states', 14),
 ('cost', 12),
 ('almost', 19)]

In [103]:
# d. sort in alphabetical order
counts_nostop.sortByKey().take(20)

[('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1),
 ('(b)', 1),
 ('(c)', 1),
 ('(does', 1),
 ('(if', 1),
 ('(japan),', 1),
 ('(or', 3),
 ('(saturday,', 1),
 ('(sort', 1),
 ('(sunday)', 1),
 ('(trademark/copyright)', 1),
 ('(www.gutenberg.org),', 1)]

In [104]:
# e. sort descending by word frequency
counts_nostop.sortBy(lambda kv: kv[1], ascending=False).take(20)

[('mr.', 373),
 ('fogg', 365),
 ('phileas', 250),
 ('passepartout', 239),
 ('fogg,', 132),
 ('fix', 129),
 ('passepartout,', 121),
 ('“i', 115),
 ('hundred', 92),
 ('replied', 89),
 ('without', 86),
 ('thousand', 83),
 ('and,', 83),
 ('time', 82),
 ('chapter', 74),
 ('train', 74),
 ('fix,', 72),
 ('going', 70),
 ('seemed', 69),
 ('great', 68)]

In [105]:
# f. remove punctuations and blank spaces
import re, string

# ASCII punctuation + typographic quotes and dashes found in the Gutenberg text
PUNCT = string.punctuation + '“”‘’—–…'
PUNCT_RE = re.compile('[' + re.escape(PUNCT) + ']')

def clean(word):
    return PUNCT_RE.sub('', word).strip()

def tokenize(line):
    # lower case + split on whitespace AND apostrophes (qu'il -> qu, il ; Fogg's -> fogg, s)
    return re.split(r"[\s'’]+", line.lower())

counts_clean = rdd.flatMap(tokenize) \
                  .map(clean) \
                  .filter(lambda w: w != '' and w not in STOPWORDS) \
                  .map(lambda w: (w, 1)) \
                  .reduceByKey(lambda a, b: a + b) \
                  .sortBy(lambda kv: kv[1], ascending=False)
counts_clean.take(20)

[('fogg', 645),
 ('passepartout', 422),
 ('fix', 256),
 ('phileas', 255),
 ('aouda', 136),
 ('master', 128),
 ('time', 125),
 ('train', 119),
 ('sir', 100),
 ('hundred', 98),
 ('replied', 93),
 ('steamer', 91),
 ('hours', 89),
 ('without', 88),
 ('thousand', 88),
 ('day', 85),
 ('days', 84),
 ('man', 77),
 ('going', 76),
 ('left', 76)]

### All transformations chained in a single function

In [106]:
def word_count(rdd, stopwords=STOPWORDS):
    """Full pipeline: words -> lower case -> no punctuation -> no stop words
    -> count -> sort by decreasing frequency. Returns an RDD of (word, count)."""
    return (rdd
            .flatMap(tokenize)                                # b. lower case + split
            .map(clean)                                       # f. punctuation
            .filter(lambda w: w != '' and w not in stopwords) # c. + f. stop words / blanks
            .map(lambda w: (w, 1))                            # (word, 1)
            .reduceByKey(lambda a, b: a + b)                  # a. count
            .sortBy(lambda kv: kv[1], ascending=False))       # e. descending sort

top_en = word_count(rdd)
top_en.take(20)

[('fogg', 645),
 ('passepartout', 422),
 ('fix', 256),
 ('phileas', 255),
 ('aouda', 136),
 ('master', 128),
 ('time', 125),
 ('train', 119),
 ('sir', 100),
 ('hundred', 98),
 ('replied', 93),
 ('steamer', 91),
 ('hours', 89),
 ('without', 88),
 ('thousand', 88),
 ('day', 85),
 ('days', 84),
 ('man', 77),
 ('going', 76),
 ('left', 76)]

In [107]:
# d. the same, sorted alphabetically
word_count(rdd).sortByKey().take(20)

[('07042', 1),
 ('1', 5),
 ('1000', 1),
 ('103', 1),
 ('10th', 1),
 ('11', 1),
 ('1140', 1),
 ('117', 2),
 ('1170', 1),
 ('11th', 7),
 ('12th', 3),
 ('13', 2),
 ('13th', 3),
 ('14th', 4),
 ('158½', 1),
 ('15th', 1),
 ('16th', 1),
 ('1756', 1),
 ('17th', 2),
 ('1814', 1)]

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [108]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # the key stay the same and the value became a tuple
  .map(lambda x: (x[0], (x[1], 1)))
  # For every key that have the same value spark add them to the other.
  # The unique value stays alone
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # The sum divided by the number of element added of the same value.
  # If it's alone it's divided by 1
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

agesRDD.collect()

[('Brooke', 22.5), ('Denny', 31.0), ('Jules', 30.0), ('TD', 35.0)]

## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible

In [109]:
import time

def timeit(label, make_rdd, n=3):
    """Times an action (count) on the RDD returned by make_rdd, n times, and prints
    the best run. count() forces the whole pipeline to be computed."""
    best = min(_time_once(make_rdd) for _ in range(n))
    print(f"{label:<45} {best:6.2f} s")
    return best

def _time_once(make_rdd):
    t0 = time.perf_counter()
    make_rdd().count()
    return time.perf_counter() - t0


# Order 1: count FIRST, clean AFTER (bad)
def pipeline_count_first():
    return (rdd.flatMap(lambda l: l.split(' '))
               .map(lambda w: (w, 1))
               .reduceByKey(lambda a, b: a + b)        # shuffle on raw words
               .map(lambda kv: (clean(kv[0].lower()), kv[1]))
               .filter(lambda kv: kv[0] != '' and kv[0] not in STOPWORDS)
               .reduceByKey(lambda a, b: a + b))       # 2nd shuffle to merge keys again

# Order 2: clean FIRST, count AFTER (optimal)
def pipeline_clean_first():
    return word_count(rdd)

# Order 3: same as 2, without the final sort (sorting is one more shuffle)
def pipeline_clean_first_nosort():
    return (rdd.flatMap(tokenize)
               .map(clean)
               .filter(lambda w: w != '' and w not in STOPWORDS)
               .map(lambda w: (w, 1))
               .reduceByKey(lambda a, b: a + b))

timeit("1. count -> clean -> count (2 shuffles)", pipeline_count_first)
timeit("2. clean -> count -> sort", pipeline_clean_first)
_ = timeit("3. clean -> count (no sort)", pipeline_clean_first_nosort)

1. count -> clean -> count (2 shuffles)         0.23 s
2. clean -> count -> sort                       0.37 s
3. clean -> count (no sort)                     0.16 s


First pipeline we count then clean, it works but we did two shuffle instead of one so we can upgrade

Second one We clean first then we count and we classify them, narrow reduce the volume before the shuffle of reduceByKey, then we classify them by using sortBy clean output but expensive for the computer.

Finally we just clean and count might be the best one and the quick one

## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [110]:
!wget -nc -O le_tour_du_monde_en_80_jours.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

File ‘le_tour_du_monde_en_80_jours.txt’ already there; not retrieving.


In [111]:
STOPWORDS_FR = {
    'le', 'la', 'les', 'l', 'un', 'une', 'des', 'du', 'de', 'd', 'et', 'ou', 'mais',
    'à', 'au', 'aux', 'en', 'dans', 'sur', 'sous', 'par', 'pour', 'avec', 'sans',
    'ce', 'cet', 'cette', 'ces', 'se', 's', 'sa', 'son', 'ses', 'leur', 'leurs',
    'il', 'elle', 'ils', 'elles', 'je', 'tu', 'nous', 'vous', 'on', 'y', 'ne', 'n',
    'pas', 'plus', 'que', 'qu', 'qui', 'quoi', 'dont', 'où', 'est', 'était', 'été',
    'être', 'a', 'ont', 'avait', 'avaient', 'fut', 'sont', 'c', 'm', 'me', 'moi', 'lui',
    'tout', 'tous', 'toute', 'toutes', 'très', 'bien', 'si', 'comme', 'même', 'aussi',
    'donc', 'alors', 'là', 'ci', 'cela', 'ça', 'mon', 'ma', 'mes', 'ton', 'ta', 'tes',
    'notre', 'votre', 'nos', 'vos', 'dit', 'répondit', 'fit', 'deux', 'peu', 'point',
    # Project Gutenberg license (in English), also present in the French file
    'project', 'gutenberg', 'ebook', 'www', 'org', 'the', 'of', 'and', 'to', 'you',
    'or', 'this', 'with', 'any', 'in', 'is', 'for', 'by', '',
}

# same pipeline, other text, other stop words: only the language changes
rdd_fr = sc.textFile('le_tour_du_monde_en_80_jours.txt')
top_fr = word_count(rdd_fr, STOPWORDS_FR)

In [112]:
# EDA: a few basic figures for each version
def stats(name, rdd_txt, top):
    n_lines = rdd_txt.count()
    n_words = top.map(lambda kv: kv[1]).sum()
    n_vocab = top.count()
    print(f"{name:<8} lines={n_lines:>6}  words (no stop words)={n_words:>7}  "
          f"vocabulary={n_vocab:>6}")

stats("english", rdd, top_en)
stats("french", rdd_fr, top_fr)

english  lines=  8312  words (no stop words)=  32751  vocabulary=  7304
french   lines=  9969  words (no stop words)=  39462  vocabulary= 10342


In [113]:
# Top 15 of each version side by side
en15, fr15 = top_en.take(15), top_fr.take(15)
print(f"{'ENGLISH':<22} {'FRENCH'}")
for (we, ce), (wf, cf) in zip(en15, fr15):
    print(f"{we:<14}{ce:>6}    {wf:<14}{cf:>6}")

ENGLISH                FRENCH
fogg             645    fogg             682
passepartout     422    passepartout     451
fix              256    phileas          330
phileas          255    mr               287
aouda            136    fix              285
master           128    heures           242
time             125    aouda            134
train            119    après            132
sir              100    mrs              131
hundred           98    quelques         129
replied           93    monsieur         123
steamer           91    maître           116
hours             89    eût              113
without           88    train            112
thousand          88    ni               109


In [114]:
# Comparison query: words present in BOTH versions (mostly proper nouns, which are
# not translated) with their number of occurrences on each side.
# join = join on the key (the word) -> (word, (count_en, count_fr))
common = (top_en.join(top_fr)
                .map(lambda kv: (kv[0], kv[1][0], kv[1][1], kv[1][0] - kv[1][1]))
                .sortBy(lambda t: t[1] + t[2], ascending=False))

print(f"{'word':<14}{'EN':>6}{'FR':>6}{'diff':>8}")
for w, c_en, c_fr, diff in common.take(20):
    print(f"{w:<14}{c_en:>6}{c_fr:>6}{diff:>+8}")

word              EN    FR    diff
fogg             645   682     -37
passepartout     422   451     -29
phileas          255   330     -75
fix              256   285     -29
aouda            136   134      +2
train            119   112      +7
sir              100    54     +46
monsieur          29   123     -94
gentleman         41    85     -44
bombay            59    61      -2
moment            50    70     -20
minutes           64    48     +16
francis           54    53      +1
steamer           91    16     +75
work              54    45      +9
without           88     8     +80
station           46    45      +1
days              84     4     +80
fort              24    59     -35
guide             41    42      -1


French has more , because french are better in litterature. I'm joking but the paging is different there are 6000 words in addition than the english version, moreover 3000 different utilisation of vocubaluray

We compare the Proper name because in english we use he or Mr. to describe someone whereas in french we directly use their name Phileas !

In [115]:
# release resources
spark.stop()